<h1 style="text-align: center; font-weight: bold;">MLP (MULTILAYER PERCEPTRON) MODEL</h1>

## Loading Dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

In [2]:
# Load proccessed dataset
data = pd.read_csv("../data/processed_train.csv")
# Load test dataset
test_data = pd.read_csv("../data/processed_test.csv")

# Drop id column
data = data.drop(columns=['id'])
# Keep test id column separately
test_ids = test_data['id']
# Drop test id column
test_data = test_data.drop(columns=['id'])

data.head()

,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Overall Stress Level,Depression
0,Aaradhya,0,49.0,Ludhiana,1,Chef,0.0,5.0,0.0,0.0,2.0,9.0,Healthy,BHM,0,1.0,2.0,0,5.0,0
1,Vivan,1,26.0,Varanasi,1,Teacher,0.0,4.0,0.0,0.0,3.0,4.0,Unhealthy,LLB,1,7.0,3.0,0,4.0,1
2,Yuvraj,1,33.0,Visakhapatnam,0,Not Applicable,5.0,0.0,5.5,2.0,0.0,5.5,Healthy,B.Pharm,1,3.0,1.0,0,4.0,1
3,Yuvraj,1,22.0,Mumbai,1,Teacher,0.0,5.0,0.0,0.0,1.0,4.0,Moderate,BBA,1,10.0,1.0,1,5.0,1
4,Rhea,0,30.0,Kanpur,1,Business Analyst,0.0,1.0,0.0,0.0,1.0,5.5,Unhealthy,BBA,1,9.0,4.0,1,4.0,0


## Define Constants

In [3]:
LABEL_COL = 'Depression'
NUMERIC_COLS = ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
                'Sleep Duration', 'Work/Study Hours', 'Financial Stress', 'Overall Stress Level']
CATEGORICAL_COLS = ['Name', 'City', 'Profession', 'Dietary Habits', 'Degree', ]

NUM_FOLD = 10

## Data Preprocessing

In [4]:
# Separate features and labels
X = data.drop(columns=[LABEL_COL])
y = data[LABEL_COL]

# Standardize numeric features
scaler = StandardScaler()
X[NUMERIC_COLS] = scaler.fit_transform(X[NUMERIC_COLS])
test_data[NUMERIC_COLS] = scaler.transform(test_data[NUMERIC_COLS])

# Encode categorical features
encoder = TargetEncoder()
X[CATEGORICAL_COLS] = encoder.fit_transform(X[CATEGORICAL_COLS], y)
test_data[CATEGORICAL_COLS] = encoder.transform(test_data[CATEGORICAL_COLS])

## Tuning Hyperparameters

In [5]:
# # Tune MLPClassifier using GridSearchCV

# # Parameters for grid search
# param_grid = {
#     'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100)],
#     'activation': ['logistic', 'relu', 'tanh'],
#     'alpha': [0.0001, 0.001, 0.01],
#     'learning_rate_init': [0.001, 0.01],
#     'batch_size': [16], # Reduce memory usage
#     'early_stopping': [True] # Prevent overfitting and possibly save time
# }

# mlp_base = MLPClassifier(max_iter=500, random_state=42)
# gs = GridSearchCV(estimator=mlp_base, param_grid=param_grid, scoring='f1', cv=NUM_FOLD, n_jobs=-1, verbose=2)
# gs.fit(X, y)

# print("Best Hyperparameters:", gs.best_params_)
# print("Best CV f1 score:", gs.best_score_)

Kaggle notebook for tuning proccedure: [Link](https://www.kaggle.com/code/tdat94/mentalhealth-mlp-training)

## K-Fold Cross Validation

In [6]:
# Best estimator from grid search
best_mlp = MLPClassifier(
    activation='relu', 
    alpha=0.0001, 
    batch_size=16,
    early_stopping=True,
    hidden_layer_sizes=(50,),
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42
)

# K-fold Cross Validation Training
kf = KFold(n_splits=NUM_FOLD, shuffle=True, random_state=42)

fold = 1 # Fold counter
classification_reports = [] # To store classification reports for each fold

for train_index, val_index in kf.split(X):
    print(f"Training fold {fold}...")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Train the model
    best_mlp.fit(X_train, y_train)
    
    # Evaluate the model
    y_pred = best_mlp.predict(X_val)
    cr = classification_report(y_val, y_pred, output_dict=True)
    cm = confusion_matrix(y_val, y_pred)
    print(f"Fold {fold} results:")
    print(cr)
    print("Confusion Matrix:")
    print(cm)
    
    # Store report
    classification_reports.append(cr)
    
    print("\n")
    fold += 1

Training fold 1...
Fold 1 results:
{'0': {'precision': 0.9560326320309146, 'recall': 0.967077831827658, 'f1-score': 0.9615235134084726, 'support': 11512.0}, '1': {'precision': 0.8437113402061855, 'recall': 0.7998436278342455, 'f1-score': 0.8211920529801324, 'support': 2558.0}, 'accuracy': 0.9366737739872069, 'macro avg': {'precision': 0.89987198611855, 'recall': 0.8834607298309518, 'f1-score': 0.8913577831943025, 'support': 14070.0}, 'weighted avg': {'precision': 0.9356120304326447, 'recall': 0.9366737739872069, 'f1-score': 0.9360105158409038, 'support': 14070.0}}
Confusion Matrix:
[[11133   379]
 [  512  2046]]


Training fold 2...
Fold 2 results:
{'0': {'precision': 0.9626339169061928, 'recall': 0.963221195746906, 'f1-score': 0.962927466782836, 'support': 11474.0}, '1': {'precision': 0.8370027037466203, 'recall': 0.8347457627118644, 'f1-score': 0.8358727097396336, 'support': 2596.0}, 'accuracy': 0.9395167022032693, 'macro avg': {'precision': 0.8998183103264066, 'recall': 0.8989834792

## Cross Validation Results

In [7]:
avg_report = {}

# Extract average metrics across folds
for key in classification_reports[0].keys():
    if key in ['0', '1', 'macro avg', 'weighted avg']: # Average the metric sections
        avg_report[key] = {}
        for metric in classification_reports[0][key].keys(): # Iterate through metrics ('precision', 'recall', etc.)
            avg_report[key][metric] = np.mean([report[key][metric] for report in classification_reports])
    elif key == 'accuracy':
        avg_report[key] = np.mean([report[key] for report in classification_reports])

# Format and print the average classification report
headers = ["precision", "recall", "f1-score", "support"]
table = []
for label in ['0', '1', 'macro avg', 'weighted avg']:
    if label in avg_report:
        row = [
            label,
            f"{avg_report[label]['precision']:.4f}",
            f"{avg_report[label]['recall']:.4f}",
            f"{avg_report[label]['f1-score']:.4f}",
            f"{int(avg_report[label]['support']):d}"
        ]
        table.append(row)

print(tabulate(table, headers=headers, floatfmt=".4f", numalign="right"))
print(f"\nAverage Accuracy: {avg_report['accuracy']:.4f}")

                precision    recall    f1-score    support
------------  -----------  --------  ----------  ---------
0                  0.9587    0.9664      0.9625      11513
1                  0.8432    0.8125      0.8275       2556
macro avg          0.9010    0.8894      0.8950      14070
weighted avg       0.9377    0.9385      0.9380      14070

Average Accuracy: 0.9385


## Train Final Model on Full Dataset

In [9]:
# Train final model on full dataset
final_model = MLPClassifier(
    hidden_layer_sizes=(50,), activation='relu', alpha=0.0001, batch_size=16,
    early_stopping=True, learning_rate_init=0.001, max_iter=500, random_state=42)

final_model.fit(X, y)

,hidden_layer_sizes,"(50,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,16
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,500
,shuffle,True
,random_state,42


In [10]:
# Predict on test data
test_preds = final_model.predict(test_data)

# Prepare submission dataframe
submission_df = pd.DataFrame({
    "id": test_ids,
    "Depression": test_preds
})

# Save submission file
submission_df.to_csv("../data/HCMUS-22KHDL-Cloud_Djata_MLP_v1.csv", index=False)